In [1]:
import pandas as pd

In [2]:
salesdf = pd.read_csv("sales_messy.csv")

In [3]:
customersdf = pd.read_csv("customers.csv")

Step 1 — Profiling Section Complete (Easy)

In [4]:
print("Shape:",salesdf.shape)

Shape: (208, 9)


In [5]:
print("\nInfo:", salesdf.info())


<class 'pandas.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     208 non-null    int64  
 1   order_date   208 non-null    str    
 2   customer_id  200 non-null    float64
 3   country      208 non-null    str    
 4   category     208 non-null    str    
 5   product      208 non-null    str    
 6   quantity     208 non-null    int64  
 7   unit_price   195 non-null    float64
 8   discount     188 non-null    float64
dtypes: float64(3), int64(2), str(4)
memory usage: 14.8 KB

Info: None


In [6]:
print("\nMissing Values:", salesdf.isnull().sum())
print("\nDuplicate Rows:", salesdf.duplicated().sum())
print("\nCountries", salesdf["country"].unique())


Missing Values: order_id        0
order_date      0
customer_id     8
country         0
category        0
product         0
quantity        0
unit_price     13
discount       20
dtype: int64

Duplicate Rows: 8

Countries <StringArray>
[     'GERMANY',      'Germany',       'France',      ' France',
   'Kazakhstan',           'UK', ' kazakhstan ',          'uk ',
       'Poland',          'usa',          'USA',       'Russia']
Length: 12, dtype: str


Step 2 — Cleaning Section Verified (Easy)

In [7]:
salesdf = salesdf.drop_duplicates()

In [8]:
salesdf["country"] = salesdf["country"].str.strip().str.title()

In [9]:
salesdf["discount"] = salesdf["discount"].fillna(0)

In [10]:
median_price = salesdf["unit_price"].median()
salesdf["unit_price"] = salesdf["unit_price"].fillna(median_price)

In [11]:
salesdf = salesdf.dropna(subset=["customer_id"])

In [12]:
salesdf["order_date"] = pd.to_datetime(salesdf["order_date"])


In [13]:
salesdf[["discount", "unit_price", "customer_id"]].isnull().sum()

discount       0
unit_price     0
customer_id    0
dtype: int64

Missing discounts were filled with 0 because a blank discount usually means no discount was applied. Missing unit prices were filled with the median because the median is less affected by extreme values than the mean. Rows without customer IDs were removed because they cannot be linked to customers.

Step 3 — Merge Without Surprises (Medium)

In [14]:
salesdf["revenue"] = (salesdf["quantity"]* salesdf["unit_price"]* (1 - salesdf["discount"]))

In [15]:
salesdf["order_date"] = pd.to_datetime(salesdf["order_date"])

In [16]:
salesdf["month"] = salesdf["order_date"].dt.month

In [17]:
salesdf["customer_id"] = salesdf["customer_id"].astype(int)
customersdf["customer_id"] = customersdf["customer_id"].astype(int)


In [18]:
before = salesdf.shape[0]


In [19]:
salesdf = salesdf.merge(customersdf, on="customer_id",how="left")

In [20]:
print("Rows before merge:", before)
print("Rows after merge:", salesdf.shape[0])


Rows before merge: 193
Rows after merge: 193


In [21]:
assert salesdf.shape[0] == before

In [22]:
salesdf.columns = salesdf.columns.str.strip().str.lower()
customersdf.columns = customersdf.columns.str.strip().str.lower()

Step 4 — Three Aggregations (Medium)

In [23]:
category_revenue = (salesdf.groupby("category")["revenue"].sum().sort_values(ascending=False).reset_index())

In [24]:
category_revenue

,category,revenue
0,Laptops,161187.4000
1,Phones,63403.4000
2,Monitors,58295.5500
3,Accessories,10323.5465


In [25]:
month_revenue = (salesdf.groupby("month")["revenue"].sum().sort_values(ascending=False).reset_index())
month_revenue

,month,revenue
0,7,42529.4260
1,10,33697.7500
2,8,30827.4315
3,4,26456.2405
4,6,24754.8375
5,5,23633.5065
6,12,22739.7510
7,3,19836.5880
8,2,19631.0780
9,11,19117.3905


In [26]:
segment_revenue = (
    salesdf.groupby("segment")["revenue"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)

segment_revenue

,segment,revenue
0,Consumer,174850.8365
1,Education,77782.0320
2,Business,40577.0280


In [29]:
total_revenue = salesdf["revenue"].sum()

Step 5 — Draft Conclusions (Hard)
Task: Write your first draft of the 3–5 findings. Each must reference a number you can point to in a table above. Mark any claim you are unsure about with TODO to verify later.

In [30]:
top_category = category_revenue.iloc[0]
top_category_share = ( top_category["revenue"] / total_revenue * 100)


In [31]:
best_month = month_revenue.iloc[0]
top_segment = segment_revenue.iloc[0]


In [32]:
print("Top category:", top_category)
print("Share:")
print(round(top_category_share, 2))
print("Best month:")
print(best_month)
print("Top segment:")
print(top_segment)

Top category: category     Laptops
revenue     161187.4
Name: 0, dtype: object
Share:
54.97
Best month:
month          7.000
revenue    42529.426
Name: 0, dtype: float64
Top segment:
segment       Consumer
revenue    174850.8365
Name: 0, dtype: object


Conclusion
1. The highest-revenue category was [Category Name], generating [Revenue], which represents [Share] of total revenue.

2. The best-performing month was [Month], producing [Revenue] in revenue.

3. The leading customer segment was [Segment], contributing [Revenue].

4. Revenue was concentrated in a small number of categories, with the top category significantly outperforming the others.

5. The difference between the best and worst month was [Difference], indicating noticeable seasonal variation in sales.